# Import and setup

In [ ]:
#Setup
!pip install --upgrade git+https://github.com/Sirius-Siru/digit_recognizer.git
!pip install -q kaggle

from google.colab import files

!rm -f kaggle.json
files.upload()

!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c digit-recognizer
!unzip -o digit-recognizer.zip -d data
!rm digit-recognizer.zip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from src.clean_data import normalize

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load data

In [5]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

#Clean data
train, test = normalize(train, test)

TypeError: cannot unpack non-iterable NoneType object

# Preprocessing

In [ ]:
# This cell use to split train-test

X = train.drop('label', axis = 1)
y = train['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train.head(5)

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
34941,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24433,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24432,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8832,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30291,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# This cell use to unflatten image
img = X.values.reshape(-1, 28, 28)

AttributeError: 'DataFrame' object has no attribute 'reshape'

In [ ]:
# This cell use to Histogram of Oriented Gradients (HOG)
def extract_hog(img_28x28):
    features, hog_image = hog(
        img_28x28,
        orientations=9,
        pixels_per_cell=(4, 4),
        cells_per_block=(2, 2),
        visualize=True,
        channel_axis=None
    )
    return features

X_hog = [extract_hog(image) for image in img]

# Model selection + Train model

In [ ]:
model = RandomForestClassifier(n_estimators = 200, oob_score=True, random_state = 42)

model.fit(X_train, y_train)

# Evaluation

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy = {accuracy*100:.2f}')
print(f'OOB accuracy: {model.oob_score_:.4f}')

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
probs = model.predict_proba(X_test)
y_pred = np.argmax(probs, axis=1)
confidence = np.max(probs, axis=1)
wrong_idx = np.where(y_pred != y_test)[0]
wrong_conf = confidence[wrong_idx]
top_wrong_idx = wrong_idx[np.argsort(-wrong_conf)]
top20 = top_wrong_idx[:20]

for i, idx in enumerate(top20):
    plt.subplot(4, 5, i+1)

    img = X_test.iloc[idx].values.reshape(28, 28)
    plt.imshow(img, cmap='gray')

    plt.title(f"True: {y_test.iloc[idx]}\nPred: {y_pred[idx]}\nConf: {confidence[idx]:.2f}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Submission